In [1]:
# ============================================================
# DLC REACH TRAJECTORY — FIRST IN-TRIAL REACHES (03/04)
# ============================================================
# Loads DLC paw kinematics, aligns to first-in-trial reach onsets,
# and plots:
#   A) 2D (x, y) paw trajectory spaghetti
#   B) x position vs time
#   C) y position vs time
#   D) paw speed vs time
# Uses canonical_time for drop-robust frame->second alignment.
# ============================================================
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from scipy.ndimage import gaussian_filter1d

print("\n===== DLC REACH TRAJECTORY (first in-trial reaches) =====")

# ------------------------------------------------------------
# USER EDIT
# ------------------------------------------------------------
analysis_folder = Path(
    r"G:\Kevin\2026-03-10_13-58-34\Record Node 114\2026-03-10_13-58-34_experiment1_recording1_analysis"
)
dlc_npz_path = Path(
    r"G:\Kevin\2026-03-10_13-58-34\Record Node 114\2026-03-10_13-58-34_experiment1_recording1_analysis"  # update to the actual path on G drive
)

PRIMARY_BODYPART  = "paw"
LIKELIHOOD_THRESH = 0.5
WIN_PRE_S         = 0.3       # seconds before reach onset
WIN_POST_S        = 1.0       # seconds after reach onset
SMOOTH_SIGMA      = 1.5       # gaussian smoothing on position (in frames; ~25 ms at 60 Hz)
FRAME_RATE        = 60.0
# ------------------------------------------------------------

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
dlc = np.load(dlc_npz_path, allow_pickle=True)
ct = dlc["canonical_time"]            # (n_frames,)  seconds
px = dlc[f"{PRIMARY_BODYPART}_x"].astype(np.float64)
py = dlc[f"{PRIMARY_BODYPART}_y"].astype(np.float64)
pl = dlc[f"{PRIMARY_BODYPART}_likelihood"]

n_frames = len(ct)
print(f"DLC frames: {n_frames}")
print(f"Canonical time range: {ct[0]:.2f} - {ct[-1]:.2f} s")
print(f"Bodypart: {PRIMARY_BODYPART}")
print(f"Frames above likelihood threshold ({LIKELIHOOD_THRESH}): "
      f"{(pl >= LIKELIHOOD_THRESH).sum()} / {n_frames} "
      f"({100*(pl >= LIKELIHOOD_THRESH).mean():.1f}%)")

# Mask low-likelihood frames with NaN BEFORE smoothing
px_masked = np.where(pl >= LIKELIHOOD_THRESH, px, np.nan)
py_masked = np.where(pl >= LIKELIHOOD_THRESH, py, np.nan)

# Smooth (mild) — using gaussian_filter1d with nan-preserving via fill+filter trick
# Actually we'll smooth per-trial below to avoid contaminating across reach boundaries.

# ------------------------------------------------------------
# LOAD REACH EVENTS (first in-trial reaches)
# ------------------------------------------------------------
bouts = np.load(analysis_folder / "peak_detection" / "bouts.npz")
onset_t = bouts["onset_t"]
trial_id = bouts["trial_id"]

first_idx = []
for tid in sorted(set(trial_id[trial_id >= 0])):
    ii = np.where(trial_id == tid)[0]
    first_idx.append(ii[np.argmin(onset_t[ii])])
first_reach_t = np.sort(onset_t[np.array(first_idx)])
print(f"First-in-trial reaches: {len(first_reach_t)}")

# ------------------------------------------------------------
# EXTRACT PER-TRIAL TRAJECTORIES (aligned via canonical_time)
# ------------------------------------------------------------
# Build a regular time axis at 60 Hz spanning the window
window_dt = 1.0 / FRAME_RATE
window_times = np.arange(-WIN_PRE_S, WIN_POST_S + window_dt/2, window_dt)
n_bins = len(window_times)

trial_x = np.full((len(first_reach_t), n_bins), np.nan)
trial_y = np.full((len(first_reach_t), n_bins), np.nan)

kept_trials = 0
for ti, t_reach in enumerate(first_reach_t):
    # Find DLC frame closest to reach onset
    onset_frame = np.searchsorted(ct, t_reach)
    # Find window bounds
    t_lo = t_reach - WIN_PRE_S
    t_hi = t_reach + WIN_POST_S
    f_lo = np.searchsorted(ct, t_lo)
    f_hi = np.searchsorted(ct, t_hi)
    if f_hi - f_lo < 5:
        continue  # skip trials with too few frames (dropped chunk?)

    # Resample to regular grid using nearest-frame lookup
    target_times = t_reach + window_times
    target_frames = np.searchsorted(ct, target_times)
    target_frames = np.clip(target_frames, 0, n_frames - 1)
    # If any target time is more than 2 frame periods off from its assigned frame, mark NaN
    actual_times = ct[target_frames]
    bad = np.abs(actual_times - target_times) > 2 * window_dt
    tx = px_masked[target_frames].copy()
    ty = py_masked[target_frames].copy()
    tx[bad] = np.nan; ty[bad] = np.nan

    # Light smoothing per-trial (skip if too few valid frames)
    n_valid = np.sum(~np.isnan(tx))
    if n_valid >= 5:
        # Use a simple fill-then-smooth pattern for nans
        for arr in (tx, ty):
            mask = ~np.isnan(arr)
            if mask.sum() >= 3:
                idx = np.where(mask)[0]
                arr[~mask] = np.interp(np.where(~mask)[0], idx, arr[mask])
        tx = gaussian_filter1d(tx, sigma=SMOOTH_SIGMA)
        ty = gaussian_filter1d(ty, sigma=SMOOTH_SIGMA)

    trial_x[ti] = tx
    trial_y[ti] = ty
    kept_trials += 1

print(f"Kept trials: {kept_trials} / {len(first_reach_t)}")

# Compute speed: |d/dt of position|, smoothed
trial_speed = np.full_like(trial_x, np.nan)
for ti in range(len(first_reach_t)):
    dx = np.gradient(trial_x[ti]) * FRAME_RATE
    dy = np.gradient(trial_y[ti]) * FRAME_RATE
    trial_speed[ti] = np.sqrt(dx**2 + dy**2)

# Means (ignoring NaN)
mean_x = np.nanmean(trial_x, axis=0)
mean_y = np.nanmean(trial_y, axis=0)
mean_speed = np.nanmean(trial_speed, axis=0)

# Identify the time-zero index for plotting paw-at-onset
t_zero_idx = np.argmin(np.abs(window_times))

# ------------------------------------------------------------
# FIGURE
# ------------------------------------------------------------
fig = plt.figure(figsize=(12, 9))
gs = GridSpec(2, 2, height_ratios=[1.4, 1.0], width_ratios=[1, 1],
              hspace=0.35, wspace=0.30)

# ---- Panel A: 2D trajectory ----
ax = fig.add_subplot(gs[0, 0])
for ti in range(len(first_reach_t)):
    ax.plot(trial_x[ti], trial_y[ti], "-", color="0.5", alpha=0.18, lw=0.8)
# Mean
ax.plot(mean_x, mean_y, "-", color="C3", lw=2.5, label="mean")
# Paw position at reach onset
ax.plot(mean_x[t_zero_idx], mean_y[t_zero_idx], "o",
        color="black", markersize=8, label="paw @ reach onset", zorder=5)
ax.invert_yaxis()  # image coord -> natural display
ax.set_xlabel("Paw x (px)")
ax.set_ylabel("Paw y (px, flipped)")
ax.set_aspect("equal")
ax.set_title(f"A. 2D paw trajectory ({kept_trials} first-in-trial reaches)",
             fontsize=11, fontweight="bold", loc="left")
ax.legend(fontsize=9, loc="best", frameon=False)
ax.spines[["top", "right"]].set_visible(False)

# ---- Panel B: x vs time ----
ax = fig.add_subplot(gs[0, 1])
for ti in range(len(first_reach_t)):
    ax.plot(window_times, trial_x[ti], "-", color="0.5", alpha=0.18, lw=0.8)
ax.plot(window_times, mean_x, "-", color="C0", lw=2.5, label="mean")
ax.axvline(0, color="black", lw=1, linestyle="--")
ax.set_xlabel("Time from reach onset (s)")
ax.set_ylabel("Paw x (px)")
ax.set_title("B. Paw x position", fontsize=11, fontweight="bold", loc="left")
ax.spines[["top", "right"]].set_visible(False)

# ---- Panel C: y vs time ----
ax = fig.add_subplot(gs[1, 0])
for ti in range(len(first_reach_t)):
    ax.plot(window_times, trial_y[ti], "-", color="0.5", alpha=0.18, lw=0.8)
ax.plot(window_times, mean_y, "-", color="C2", lw=2.5, label="mean")
ax.axvline(0, color="black", lw=1, linestyle="--")
ax.invert_yaxis()
ax.set_xlabel("Time from reach onset (s)")
ax.set_ylabel("Paw y (px, flipped)")
ax.set_title("C. Paw y position", fontsize=11, fontweight="bold", loc="left")
ax.spines[["top", "right"]].set_visible(False)

# ---- Panel D: speed vs time ----
ax = fig.add_subplot(gs[1, 1])
for ti in range(len(first_reach_t)):
    ax.plot(window_times, trial_speed[ti], "-", color="0.5", alpha=0.18, lw=0.8)
ax.plot(window_times, mean_speed, "-", color="C1", lw=2.5, label="mean")
ax.axvline(0, color="black", lw=1, linestyle="--")
ax.set_xlabel("Time from reach onset (s)")
ax.set_ylabel("Paw speed (px/s)")
ax.set_title("D. Paw speed", fontsize=11, fontweight="bold", loc="left")
ax.spines[["top", "right"]].set_visible(False)

plt.suptitle(f"Paw kinematics during first-in-trial reaches  "
             f"(n={kept_trials} trials, likelihood>={LIKELIHOOD_THRESH})",
             fontsize=12, fontweight="bold", y=1.005)

out_base = analysis_folder / "qc" / "DLC_first_reach_trajectories"
plt.savefig(f"{out_base}.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved -> {out_base.name}.png / .pdf")
print("\n===== DONE =====\n")


===== DLC REACH TRAJECTORY (first in-trial reaches) =====


PermissionError: [Errno 13] Permission denied: 'G:\\Kevin\\2026-03-10_13-58-34\\Record Node 114\\2026-03-10_13-58-34_experiment1_recording1_analysis'